# Residual-зв'язок (додавання) з нуля + backprop руками

У стилі [Andrej Karpathy — «Becoming a Backprop Ninja»](https://www.youtube.com/watch?v=q8SA3rM6ckI):
розписуємо forward по одній операції, а **backward рахуємо самі** — без PyTorch і autograd, лише numpy.

**Residual-зв'язок (skip connection у ResNet):** вихід блоку = вхід + те, що порахувала гілка:
`r = a + F(a)`. Тут побачимо, **чому** це рятує глибокі мережі від зникання градієнтів —
прямо в рядках backward.

Замість згорток — повнозв'язні шари (`X @ W`): згортка — це те саме множення на ваги, лише застосоване
в кожній позиції зображення, а логіка skip-зв'язку і його градієнтів ідентична.

Задача-іграшка: точки на площині, клас 1 — всередині кола.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

SEED = 42
rng = np.random.default_rng(SEED)

N = 200
x = rng.uniform(-1, 1, size=(N, 2))
y = ((x ** 2).sum(axis=1, keepdims=True) < 0.5).astype(np.float64)

plt.figure(figsize=(4, 4))
plt.scatter(x[:, 0], x[:, 1], c=y[:, 0], cmap="coolwarm", s=15)
plt.title("1 — всередині кола, 0 — зовні")
plt.gca().set_aspect("equal")
plt.show()
print("x:", x.shape, "y:", y.shape, "частка класу 1:", y.mean().round(2))

## Ланцюгове правило (chain rule) і локальні градієнти

Позначення як у Karpathy: `dX` — це **∂loss/∂X**, градієнт loss по змінній `X`, тієї самої форми, що й `X`.

Backward іде від loss назад. Для кожної операції `out = f(in)` знаємо `dout` (прийшов «згори») і рахуємо
`din = dout · ∂out/∂in` — це **ланцюгове правило**. Тобто кожній операції досить знати лише свою
**локальну похідну**:

| Операція (forward) | Backward | Інтуїція |
|---|---|---|
| `out = X @ W` | `dX = dout @ W.T`, `dW = X.T @ dout` | форми підказують порядок множення |
| `out = X + b` (b — рядок, broadcasting) | `dX = dout`, `db = dout.sum(axis=0)` | b додано до кожного рядка → градієнти рядків сумуються |
| `out = relu(h)` | `dh = dout * (h > 0)` | пропускає градієнт, де h > 0, і «глушить» де h ≤ 0 |
| `out = a + v` | `da = dout`, `dv = dout` | **додавання копіює градієнт** в обидві гілки |
| `out = concat([a, v])` | `da = dout[:, :H]`, `dv = dout[:, H:]` | **конкатенація розрізає градієнт** на шматки |
| loss = BCE(sigmoid(z), y), середнє по N | `dz = (sigmoid(z) - y) / N` | «наскільки p відрізняється від правди» |

**Правило розгалуження:** якщо змінна використовується **двічі** (тут `a` — і в skip, і в гілці),
її градієнт — **сума** градієнтів з обох місць. Тому в коді `da = ...`, а потім `da += ...`.

## Мережа

```
x ─► h = x @ W1 + b1 ─► a = relu(h) ─┬──────────────────────────────────┐  skip
                                     └─► u = a @ W2 + b2 ─► v = relu(u) ─┴─► r = a + v ─► z = r @ W3 + b3 ─► loss
                                         (гілка F)
```

`a` і `v` мають однакову форму `(N, H)` — без цього додати їх не можна (у ResNet для цього
бувають додаткові 1×1 згортки на skip).

Маленькі випадкові `b` замість нулів — порада Karpathy: нулі можуть «сховати» помилку в backward.

In [ ]:
H = 16


def init_params():
    rng = np.random.default_rng(SEED)
    return {
        "W1": rng.normal(size=(2, H)) * np.sqrt(2 / 2),
        "b1": rng.normal(size=H) * 0.1,
        "W2": rng.normal(size=(H, H)) * np.sqrt(2 / H),
        "b2": rng.normal(size=H) * 0.1,
        "W3": rng.normal(size=(H, 1)) * np.sqrt(1 / H),
        "b3": rng.normal(size=1) * 0.1,
    }


def forward(params, x, y):
    h = x @ params["W1"] + params["b1"]
    a = np.maximum(h, 0)
    u = a @ params["W2"] + params["b2"]
    v = np.maximum(u, 0)
    r = a + v
    z = r @ params["W3"] + params["b3"]
    # Стабільна форма BCE з логітів: без exp від великих чисел.
    loss = np.mean(np.maximum(z, 0) - z * y + np.log1p(np.exp(-np.abs(z))))
    return loss, {"x": x, "h": h, "a": a, "u": u, "v": v, "r": r, "z": z}


params = init_params()
loss, cache = forward(params, x, y)
for name, value in cache.items():
    print(f"{name}: {value.shape}")
print("loss:", round(loss, 4))

## Backward — рядок за рядком

Ідемо від loss назад, по одній операції (див. таблицю вище):

1. `dz = (sigmoid(z) - y) / N` — градієнт BCE по логітах.
2. `z = r @ W3 + b3` → `dW3`, `db3`, `dr`.
3. **`r = a + v` → `da = dr`, `dv = dr`.** Додавання копіює градієнт у skip і в гілку.
4. `v = relu(u)` → `du`; `u = a @ W2 + b2` → `dW2`, `db2` і **ще один** внесок у `da`.
5. **`da` = (зі skip) + (з гілки)** — правило розгалуження: `a` використано двічі.
6. `a = relu(h)` → `dh`; `h = x @ W1 + b1` → `dW1`, `db1`.

Головне — крок 3: **через skip градієнт `dr` потрапляє в `da` без жодних множень**. Хоч би що робила
гілка (навіть якщо її градієнт майже нуль), `da` отримує щонайменше `dr`.

In [ ]:
def backward(params, cache, y):
    x, h, a, u, v, r, z = (cache[k] for k in ["x", "h", "a", "u", "v", "r", "z"])
    n = len(y)

    dz = (1 / (1 + np.exp(-z)) - y) / n
    dW3 = r.T @ dz
    db3 = dz.sum(axis=0)
    dr = dz @ params["W3"].T

    da = dr.copy()
    dv = dr

    du = dv * (u > 0)
    dW2 = a.T @ du
    db2 = du.sum(axis=0)
    da += du @ params["W2"].T

    dh = da * (h > 0)
    dW1 = x.T @ dh
    db1 = dh.sum(axis=0)
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2, "W3": dW3, "b3": db3}


grads = backward(params, cache, y)
for name, grad in grads.items():
    print(f"d{name}: {grad.shape} (як у {name}: {params[name].shape})")

## Перевірка: числовий градієнт (gradient check)

Karpathy перевіряє свої градієнти через PyTorch autograd. Ми обійдемось без бібліотек —
**числовим градієнтом** за означенням похідної (центральна різниця):

∂loss/∂w ≈ (loss(w + ε) − loss(w − ε)) / 2ε, ε = 1e-6

Для кожного числа в кожній матриці ваг трохи зсуваємо його вгору і вниз і дивимось, як змінився loss.
Це повільно (2 forward на кожну вагу), але тривіально правильно — ідеальний «еталон» для перевірки.
Усе в `float64`: у `float32` різниця loss(w+ε) − loss(w−ε) губиться в округленні.

Відносна похибка ~1e-7 і менше — градієнт правильний; ~1e-2 і більше — помилка в backward.

In [ ]:
def numerical_grad(params, name, eps=1e-6):
    param = params[name]
    grad = np.zeros_like(param)
    for idx in np.ndindex(param.shape):
        old = param[idx]
        param[idx] = old + eps
        loss_plus, _ = forward(params, x, y)
        param[idx] = old - eps
        loss_minus, _ = forward(params, x, y)
        param[idx] = old
        grad[idx] = (loss_plus - loss_minus) / (2 * eps)
    return grad


def cmp(name, manual, numeric):
    max_diff = np.abs(manual - numeric).max()
    rel_error = max_diff / (np.abs(manual).max() + np.abs(numeric).max() + 1e-12)
    status = "OK" if rel_error < 1e-6 else "ПОМИЛКА"
    print(f"{name:3s} | max різниця {max_diff:.2e} | відносна похибка {rel_error:.2e} | {status}")


for name in params:
    cmp(name, grads[name], numerical_grad(params, name))

**Експеримент для самоперевірки:** закоментуй рядок `da += du @ params["W2"].T` (забути правило
розгалуження — типова помилка) і запусти backward і перевірку знову. Які градієнти стануть неправильними, а які ні?
Чому саме вони?

## Навчання: градієнтний спуск руками

Тепер, коли градієнти правильні, навчання — це просто цикл: forward → backward → `param -= lr * grad`
(звичайний градієнтний спуск, без оптимізаторів з бібліотек). Потім малюємо межу рішень (decision boundary):
де мережа каже «всередині кола».

In [ ]:
LR = 0.5
STEPS = 3000

for step in range(STEPS + 1):
    loss, cache = forward(params, x, y)
    grads = backward(params, cache, y)
    for name in params:
        params[name] -= LR * grads[name]
    if step % 500 == 0:
        accuracy = ((cache["z"] > 0) == (y > 0.5)).mean()
        print(f"крок {step:4d}: loss {loss:.4f}, точність {accuracy:.1%}")

In [ ]:
grid_1d = np.linspace(-1, 1, 200)
grid = np.array(np.meshgrid(grid_1d, grid_1d)).reshape(2, -1).T
_, grid_cache = forward(params, grid, np.zeros((len(grid), 1)))
inside = (grid_cache["z"] > 0).reshape(200, 200)

plt.figure(figsize=(4, 4))
plt.contourf(grid_1d, grid_1d, inside, alpha=0.3, cmap="coolwarm")
plt.scatter(x[:, 0], x[:, 1], c=y[:, 0], cmap="coolwarm", s=15)
plt.gca().set_aspect("equal")
plt.title("межа рішень після навчання")
plt.show()

## Чому residual рятує глибокі мережі

Візьмемо глибоку мережу з L блоків. Звичайна (plain): `a_{l+1} = tanh(a_l @ W_l)`.
Residual: `a_{l+1} = a_l + tanh(a_l @ W_l)`. Backward одного блоку:

- plain: `da_l = (da_{l+1} * tanh') @ W_l.T`
- residual: `da_l = da_{l+1} + (da_{l+1} * tanh') @ W_l.T`

У plain-мережі градієнт **множиться** на `tanh' · W` у кожному блоці. Якщо цей множник трохи менший
за 1 — після 50 блоків від градієнта нічого не лишається: **vanishing gradients** (зникання градієнтів),
перші шари не вчаться. У residual є доданок `da_{l+1}` — «магістраль», по якій градієнт іде назад
без множень.

Нижче — норма градієнта `‖da_l‖` на вході кожного блоку, мережа з 50 блоків, ваги N(0, 0.8²/H).

In [ ]:
def gradient_norms(depth, residual, std, width=32, n=64):
    rng = np.random.default_rng(SEED)
    weights = [rng.normal(size=(width, width)) * std / np.sqrt(width) for _ in range(depth)]
    a = rng.normal(size=(n, width))
    branch_outputs = []
    for W in weights:
        t = np.tanh(a @ W)
        branch_outputs.append(t)
        a = a + t if residual else t

    da = np.ones_like(a) / n
    norms = []
    for W, t in zip(reversed(weights), reversed(branch_outputs)):
        da_branch = (da * (1 - t ** 2)) @ W.T
        da = da + da_branch if residual else da_branch
        norms.append(np.linalg.norm(da))
    return norms[::-1]


DEPTH = 50
plain = gradient_norms(DEPTH, residual=False, std=0.8)
res = gradient_norms(DEPTH, residual=True, std=0.8)

plt.figure(figsize=(8, 4))
plt.semilogy(range(DEPTH), plain, label="plain: a = tanh(a @ W)")
plt.semilogy(range(DEPTH), res, label="residual: a = a + tanh(a @ W)")
plt.xlabel("номер блоку (0 — найближчий до входу)")
plt.ylabel("‖da‖ (логарифмічна шкала)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print(f"перший блок: plain {plain[0]:.1e}, residual {res[0]:.1e}")

Plain-мережа: градієнт на першому блоці в сотні тисяч разів менший, ніж на останньому — перші шари
майже не вчаться. Residual: градієнт не зникає, а навіть **росте** до входу (тут у ~20 разів за 50 блоків) (кожен блок додає свій внесок до
«магістралі»), тому в справжніх ResNet ще є BatchNorm, а гілки ініціалізують так, щоб на старті вони
давали майже нуль.

Спробуй `std=1.0` для plain: градієнт уже не зникає — при «ідеальній» ініціалізації глибока plain-мережа
теж може вчитись. Але це баланс на лезі ножа: трохи менше — зникає, трохи більше — вибухає.
Residual-зв'язок робить мережу **стійкою** до цього.
